In [1]:
# Computes stochastic closure certificate using SOS for stochastic systems in verifying LTL specifications
# Gambler’s Ruin Example

# include important libraries
using JuMP
using MosekTools
using DynamicPolynomials
using MultivariatePolynomials
using LinearAlgebra
using TSSOS # important for SOS, see https://github.com/wangjie212/TSSOS
using Distributions # for the noise

@polyvar x y x0 # global vars used in monomials
var = [x, y]
sos_tol = 1 # the maximum degree of unknown SOS lagrange polynomials = deg + sos_tol 
error = 2   # precision digit places
gamma, lamda, delta = 1.008, 3.05, 0.1

tau1, tau2 = 0.1, 0.05 # S-procedure constants in condition 3

# returns 1 with 51% chance, -1 with 49% chance
wp, wm, pr, qr = 1, -1, 0.51, 0.49

# access odd priorities
function odd_pri(d::Dict)
    return [v for v in values(d) if isodd(v)]
end

# creates the parametrized SCC
function add_paramC_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="c_$(q)_$(p)")
    C_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return C_qp, coeffs, basis
end

function add_paramCf_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="cf_$(q)_$(p)")
    Cf_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return Cf_qp, coeffs, basis
end

# creates the parametrized ranking functions
function add_paramV_poly!(model, var2, deg, l, q, p)
    basis = monomials(var2, 0:deg)
    coeffs = @variable(model, [1:length(basis)], base_name="v_$(l)_$(q)_$(p)")
    V_l_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return V_l_qp, coeffs, basis
end

# the domain is function pairwise
# X0=[10]
gX = [x-0, y-0]
g5 = [x0-10, 10-x0, x-0, y-0]

# polynomial stochastic closure certificate of degree deg
function scc(deg, S, pri)
    # synthesize SCC by using the standard formulation
    # deg: degree of scc template
    model = Model(optimizer_with_attributes(Mosek.Optimizer))
    set_optimizer_attribute(model, MOI.Silent(), true)
    
    C_dict = Dict()  # key => (symbolic_poly, numeric_poly)
    for q in keys(S1) # DPA states
        for (lab, qn) in S[q]
            C, Cc, Cb = add_paramC_poly!(model, var, deg, q, qn) # CC, CC coefficients, and CC basis functions
            Cf, Ccf, Cbf = add_paramCf_poly!(model, var, deg, q, qn)
            key_c = (q, qn, lab, "c") 
            key_cf = (q, qn, lab, "cf")
            C_dict[key_c] = (C, Cc, Cb)  # store symbolic, coefficients, basis
            C_dict[key_cf] = (Cf, Ccf, Cbf)

            # cond 1
            C_plus1 = C([x, y]=>[x, x + wp])
            C_minus1 = C([x, y]=>[x, x - wm])
            E_C1 = pr*C_plus1 + qr*C_minus1
            add_psatz!(model, -E_C1 + gamma, [x], [x-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 

            # cond 2
            if pri[q] <= pri[qn]
                C_plus2 = Cf([x, y]=>[x, x + wp])
                C_minus2 = Cf([x, y]=>[x, x - wm])
                E_Cf2 = pr*C_plus2 + qr*C_minus2
                add_psatz!(model, -E_Cf2 + gamma, [x], [x-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
            end

            for p in keys(S1)
                # cond 3
                Cf3e, Ccf3e, Cbf3e = add_paramCf_poly!(model, var, deg, p, qn) # e as in E[.\.]
                Cf3, Ccf3, Cbf3 = add_paramCf_poly!(model, var, deg, p, q) 
                key_cf3 = (p, q, "cf") 
                key_cf3e = (p, qn, "cf")
                C_dict[key_cf3] = (Cf3, Ccf3, Cbf3)  
                C_dict[key_cf3e] = (Cf3e, Ccf3e, Cbf3e)
                if pri[p] <= pri[q] && pri[p] <= pri[qn] 
                    Cf_plus3 = Cf3e([x, y]=>[x, y + wp])
                    Cf_minus3 = Cf3e([x, y]=>[x, y - wm])
                    E_Cf3 = pr*Cf_plus3 + qr*Cf_minus3
                    add_psatz!(model, Cf3 - E_Cf3, [x, y], [x-0, y-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)              
                end

                # cond 4
                C4e, Cc4e, Cb4e = add_paramC_poly!(model, var, deg, p, qn) # e as in E[.\.]
                C4, Cc4, Cb4 = add_paramC_poly!(model, var, deg, p, q) 
                key_c4 = (p, q, "c") 
                key_c4e = (p, qn, "c")
                C_dict[key_c4] = (C4, Cc4, Cb4)  
                C_dict[key_c4e] = (C4e, Cc4e, Cb4e)
                C_plus4 = C4e([x, y]=>[x, y + wp])
                C_minus4 = C4e([x, y]=>[x, y - wm])
                E_C4 = pr*C_plus4 + qr*C_minus4
                add_psatz!(model, C4 - E_C4, [x, y], [x-0, y-0], [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 

                #### non-negativity of C and Cf
                add_psatz!(model, C4, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
                add_psatz!(model, Cf3, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                
                # cond 5
                for l in odd_pri(pri)
                    V2, Vc2, Vb2 = add_paramV_poly!(model, [x0, y], deg, l, "q0", q)
                    V1, Vc1, Vb1 = add_paramV_poly!(model, [x0, x], deg, l, "q0", p)
                    key_v1 = (l, "q0", p, "v")
                    key_v2 = (l, "q0", q, "v")
                    C_dict[key_v1], C_dict[key_v2] = (V1, Vc1, Vb1), (V2, Vc2, Vb2)
                    #### non-negativity of V
                    add_psatz!(model, V1, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                    add_psatz!(model, V2, var, gX, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
            
                    C1 = C4([x, y]=>[x0, x])
                    if l <= pri[p] && pri[p] <= pri[q]
                        add_psatz!(model, -V2 + V1 - delta + tau1*(lamda - C1) + tau2*(lamda - Cf3), [x0,x,y], g5, [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                    end
                end
            end
        end
    end
        
    optimize!(model) # solve for coefficients
    status = termination_status(model)
    C_eval_dict = Dict() # Get numerical values of coefficients and plug into polynomials
    for (key, (C, Cc, Cb)) in C_dict
        coeff_vals = round.(value.(Cc); digits=error)  # Round each coefficient to 2 decimal places
        C_numeric = sum(coeff_vals[i] * Cb[i] for i in eachindex(Cb))
        C_eval_dict[key] = (C, C_numeric)
    end
    
    # status might be optimal but if all Bc approx 10^{-error}, it's essentially 0 so OPTIMAL != CC.
    p_sat = 1-gamma/lamda # satisfaction probability lower bound
    return status, C_eval_dict, p_sat
end

# the considered DPA 

# A representing GFa
S1 = Dict(
    :"q0" => [("b", :"q0"), ("a", :"q1")],
    :"q1" => [("a", :"q1"), ("b", :"q0")]
)

pri1 = Dict(:"q0" => 1, :"q1" => 2) # priority function


# Simulation
deg = 1 # desired degree of SCC
stats = @timed (status, CC_data, p_sat) = scc(deg, S1, pri1)

println("poly deg: ", deg)
println("status: ", status)
println("sat probability: ", p_sat)
println("Number of SCC polynomials: ", length(CC_data))
println("time: ", stats.time, "\n")

for (k, v) in CC_data
    println(k, ": ", v[2], ",")
end

println("Finished")

poly deg: 1
status: OPTIMAL
sat probability: 0.6695081967213115
Number of SCC polynomials: 18
time: 10.122475292

("q0", "q1", "cf"): 0.68 - 0.92*y + 0.9*x,
("q0", "q1", "c"): -0.02 - 0.09*y + 0.9*x,
("q1", "q1", "a", "cf"): 0.15 + 0.07*y - 0.07*x,
(1, "q0", "q0", "v"): 1.42 + 0.83*x0,
("q0", "q0", "b", "c"): 0.15 + 0.07*y - 0.07*x,
("q1", "q0", "b", "cf"): 0.0,
("q1", "q0", "b", "c"): 0.15 + 0.07*y - 0.07*x,
("q1", "q0", "c"): 0.98 + 0.97*y + 0.9*x,
("q1", "q1", "a", "c"): 0.15 + 0.07*y - 0.07*x,
("q0", "q1", "a", "c"): 0.15 + 0.07*y - 0.07*x,
("q1", "q1", "cf"): 0.0,
(1, "q0", "q1", "v"): 0.76 + 0.76*x0,
("q0", "q1", "a", "cf"): 0.15 + 0.07*y - 0.07*x,
("q0", "q0", "b", "cf"): 0.15 + 0.07*y - 0.07*x,
("q0", "q0", "c"): 0.97 + 0.96*y + 0.9*x,
("q1", "q0", "cf"): 0.97,
("q1", "q1", "c"): -0.02 - 0.09*y + 0.9*x,
("q0", "q0", "cf"): 0.92 + 0.9*x,
Finished


#### To post verify the SOS SC2 via Z3 and remember to change the kernel to python

In [21]:
# SC^2 for Gambler's ruin without DPA

from z3 import *
import itertools
import numpy as np
import time
from itertools import product

# --- define symbolic polynomial variables ---
x0, x, y = Reals('x0 x y')

############################ Dynamics ###############
# SC2 from SOS
D = {
# ("q0", "q1", "cf"): 0.68 - 0.92*y + 0.9*x,
("q0", "q1", "c"): -0.02 - 0.09*y + 0.9*x,
("q1", "q1", "a", "cf"): 0.15 + 0.07*y - 0.07*x,
(1, "q0", "q0", "v"): 1.42 + 0.83*x0,
("q0", "q0", "b", "c"): 0.15 + 0.07*y - 0.07*x,
("q1", "q0", "b", "cf"): 0.0,
("q1", "q0", "b", "c"): 0.15 + 0.07*y - 0.07*x,
("q1", "q0", "c"): 0.98 + 0.97*y + 0.9*x,
("q1", "q1", "a", "c"): 0.15 + 0.07*y - 0.07*x,
("q0", "q1", "a", "c"): 0.15 + 0.07*y - 0.07*x,
("q1", "q1", "cf"): 0.0,
(1, "q0", "q1", "v"): 0.76 + 0.76*x0,
("q0", "q1", "a", "cf"): 0.15 + 0.07*y - 0.07*x,
("q0", "q0", "b", "cf"): 0.15 + 0.07*y - 0.07*x,
("q0", "q0", "c"): 0.97 + 0.96*y + 0.9*x,
# ("q1", "q0", "cf"): 0.97,
# ("q1", "q1", "c"): -0.02 - 0.09*y + 0.9*x,
# ("q0", "q0", "cf"): 0.92 + 0.9*x
}

# --- parameters ---
gamma, lamda, delta, tau1, tau2 = 1.008, 3.05, 0.1, 0.1, 0.05 

def f(x, w):
    xn = x + w
    xn_clipped = If(xn<=0, x, xn)
    return If(x == 0, 0, xn_clipped)


S1 = {
    "q0": [("b", "q0"), ("a", "q1")],
    "q1": [("a", "q1"), ("b", "q0")]
}

pri = {"q0": 1, "q1": 2}

pr = RealVal(0.51)
qr = RealVal(0.49)
wp = IntVal(1)   # w = +1
wm = IntVal(-1)  # w = -1
########################################################

odd_pri = [v for v in pri.values() if v % 2 == 1] # odd priorities

def to_z3(e): # convert SCCs to z3 liftable expressions
    return e if is_expr(e) else RealVal(e)

# --- Constraints on domains ---
s = Solver()

for q in S1.keys():
    for (lab, qn) in S1[q]:
        # cond 1
        key1 = (q,qn,lab,"c")
        if key1 in D:
            C1 = to_z3(D[key1])
            C_plus1, C_minus1  = substitute(C1, (y, f(x, wp))), substitute(C1, (y, f(x, wm)))
            E_C1 = pr*C_plus1 + qr*C_minus1
            s.add(ForAll(x, Implies(x >= 0, E_C1 <= gamma)))

        # cond 2
        key2 = (q,qn,lab,"cf")
        if key2 in D:
            Cf2 = to_z3(D[key2])
            C_plus2, C_minus2  = substitute(Cf2, (y, f(x, wp))), substitute(Cf2, (y, f(x, wm)))
            E_Cf2 = pr*C_plus2 + qr*C_minus2
            if pri[q] <= pri[qn]:
                s.add(ForAll(x, Implies(x >= 0, E_Cf2 <= gamma)))

        for p in S1.keys():
            # cond 3
            key3, key3e = (p,q,"cf"), (p,qn,"cf")
            if key3 not in D or key3e not in D:
                continue   # skip this iteration
            Cf3, Cf3e = to_z3(D[key3]), to_z3(D[key3e]) ######
            s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), Cf3 >= 0)))
            if pri[p] <= pri[q] and pri[p] <= pri[qn]:
                Cf_plus3  = substitute(Cf3e, [(y, f(y, wp))])
                Cf_minus3 = substitute(Cf3e, [(y, f(y, wm))])
                E_Cf3 = pr*Cf_plus3 + qr*Cf_minus3
                s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), E_Cf3 <= Cf3)))

            # cond 4
            key4, key4e = (p,q,"c"), (p,qn,"c")
            if key4 not in D or key4e not in D:
                continue   # skip this iteration
            C4, C4e = to_z3(D[key4]), to_z3(D[key4e]) ########
            s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), C4 >= 0)))
            C_plus4  = substitute(C4e, [(y, f(y, wp))])
            C_minus4 = substitute(C4e, [(y, f(y, wm))])
            E_C4 = pr*C_plus4 + qr*C_minus4
            s.add(ForAll([x, y], Implies(And(x >= 0, y >= 0), E_C4 <= C4)))

            # cond 5
            for l in odd_pri:
                key5a, key5b = (l,"q0",p,"v"), (l,"q0",q,"v")
                if key5a in D and key5b in D:
                    V5a = D[key5a]
                    V5b = D[key5b]
                    if l <= pri[p] and pri[p] <= pri[q]:
                        # pass
                        s.add(ForAll([x0, x, y], Implies(And(x0 == 10, x > 0, y > 0), V5a >= 0)))
                        s.add(ForAll([x0, x, y], Implies(And(x0 == 10, x > 0, y > 0), V5b >= 0)))
                        s.add(V5b <= V5a - delta + tau1*(lamda - C4) + tau2*(lamda - Cf3))
                        # s.add(Implies(And(C4 <= lamda, Cf3 <= lamda), V5b <= V5a - delta)) 

start_solve = time.time()
result = s.check()
print("sat" if result == sat else "unsat")
end_solve = time.time()
print("Solve time:", end_solve - start_solve)

sat
Solve time: 0.008367776870727539


#### For synthesis

In [1]:
# This is not for the paper
# SC^3 for Gambler's ruin without DPA

from z3 import *
import itertools
import numpy as np
import time
from itertools import product

d = 1  # polynomial degree
bd = 15
X = list(range(0, bd + 1))
X0 = [10]
Xinf = [0] # x==0
X_c = list(range(1, bd + 1)) # X/Xinf

U = [-.25, -.2, -.15, -.1, -.05, 0 , .05, .1, .15, .2, .25]

# Dynamics
def f(x, w):
    xn = x + w
    xn_clipped = If(Or(xn > -bd, xn < bd), x, xn)
    return If(x == 0, 0, xn_clipped)

pr = RealVal(0.51)
qr = RealVal(0.49)
wp = IntVal(1)   # w = +1
wm = IntVal(-1)  # w = -1

# --- define symbolic polynomial of degree d ---
x, y = Reals('x y')

def monomials(vars, degree):
    result = []
    for deg in range(degree + 1):
        for exponents in itertools.product(range(deg+1), repeat=len(vars)):
            if sum(exponents) == deg:
                term = 1
                for var, exp in zip(vars, exponents):
                    term *= var**exp
                result.append(term)
    return result

monoms = monomials([x, y], d)

# Define polynomials C, C_f and V_l
coeffs = [Real(f'c_{i}') for i in range(len(monoms))]
C = sum(c*m for c,m in zip(coeffs, monoms))
coeffs_f = [Real(f'cf_{i}') for i in range(len(monoms))]
Cf = sum(c*m for c,m in zip(coeffs_f, monoms))
coeffs_v = [Real(f'v_{i}') for i in range(len(monoms))]
V = sum(v*m for v,m in zip(coeffs_v, monoms))

# --- Unknown parameters ---
delta = Real('delta')
gamma = Real('gamma')
lamda = Real('lamda')
rho = [Real(f'rho_{i}') for i in range(len(U))]

start = time.time()

# --- Constraints on domains ---
s = Solver()
s.add(sum(rho) == 1)

for x_val in X:
    # cond 1
    C_plus1  = substitute(C, [(y, f(x, wp))])
    C_minus1 = substitute(C, [(y, f(x, wm))])
    E_C1 = sum(rho[i]*((RealVal(0.5)+U[i])*C_plus1 + (RealVal(0.5)-U[i])*C_minus1) for i in range(len(U)))
    C1 = substitute(E_C1, (x, RealVal(x_val)))
    s.add(C1 <= gamma)
    s.add(C1 >= 0)
    for y_val in X:
        # cond 3
        if x_val in X_c and y_val in X_c:
            for x0 in X0:
                C3 = substitute(C, [(x, RealVal(x0)), (y, RealVal(x_val))])
                Cf3 = substitute(Cf, [(x, RealVal(x_val)), (y, RealVal(y_val))])
                V3a = substitute(V, [(x, RealVal(x0)), (y, RealVal(x_val))])
                V3b = substitute(V, [(x, RealVal(x0)), (y, RealVal(y_val))])
                s.add(Implies(And(C3 <= lamda, Cf3 <= lamda), V3b <= V3a - delta)) 
                s.add(V3a >= 0)
                s.add(V3b >= 0)
                s.add(C3 >= 0)
                s.add(Cf3 >= 0)
        # cond 2
        for u in U:
            C2 = substitute(C, [(x, RealVal(y_val)), (y, RealVal(y_val))])
            C_plus2  = substitute(C2, [(y, f(y, wp))])
            C_minus2 = substitute(C2, [(y, f(y, wm))])
            E_C2 = (RealVal(0.5)+u)*C_plus2 + (RealVal(0.5)-u)*C_minus2
            EC2 = substitute(E_C2, (x, RealVal(x_val)), (y, RealVal(y_val)))
            const_u = EC2 <= lamda

            if x_val in X_c and y_val in X_c:

                C2a = substitute(Cf, [(x, RealVal(y_val)), (y, RealVal(y_val))])
                C_plus2a  = substitute(C2a, [(y, f(y, wp))])
                C_minus2a = substitute(C2a, [(y, f(y, wm))])
                E_C2a = (RealVal(0.5)+u)*C_plus2a + (RealVal(0.5)-u)*C_minus2a
                EC2a = substitute(E_C2a, (x, RealVal(x_val)), (y, RealVal(y_val)))
                s.add(Implies(const_u, EC2a <= gamma)) 
    
                C2b = substitute(Cf, [(x, RealVal(x_val)), (y, RealVal(y_val))])
                C_plus2b  = substitute(Cf, [(y, f(y, wp))])
                C_minus2b = substitute(Cf, [(y, f(y, wm))])
                E_C2b = (RealVal(0.5)+u)*C_plus2b + (RealVal(0.5)-u)*C_minus2b
                EC2b = substitute(E_C2b, (x, RealVal(x_val)), (y, RealVal(y_val)))
                s.add(Implies(const_u, EC2b <= gamma)) 
                                
            C2c = substitute(C, [(x, RealVal(x_val)), (y, RealVal(y_val))])
            C_plus2c  = substitute(C, [(y, f(y, wp))])
            C_minus2c = substitute(C, [(y, f(y, wm))])
            E_Cf2c = (RealVal(0.5)+u)*C_plus2c + (RealVal(0.5)-u)*C_minus2c
            EC2c = substitute(E_Cf2c, (x, RealVal(x_val)), (y, RealVal(y_val)))
            s.add(C2c >= 0) # non neg cond
            s.add(Implies(const_u, EC2c <= C2c)) 

s.add(And(0 <= gamma, gamma < lamda - RealVal(1000) ))
# s.add(And(0 <= gamma, gamma < lamda ))
s.add(gamma <= 10000)
s.add(delta > 0)
end = time.time()

print("Constraint construction time:", end - start)
def get_float(m, var, prec=10):
    return float(m.eval(var, model_completion=True).as_decimal(prec).replace("?", ""))

start_solve = time.time()
if s.check() == sat:
    m = s.model()
    print("Solution found for coefficients:")
    
    # Evaluate the unknown parameters
    delta1 = get_float(m, delta)
    gamma1 = get_float(m, gamma)
    lamda1 = get_float(m, lamda)
    print("delta =", delta1)
    print("gamma =", gamma1)
    print("lamda =", lamda1)
    print("sat_prob =", 1 - gamma1/lamda1)

    
    print(f"\nC coefficients:")
    for c in coeffs:
        print(f"{c} =", m.eval(c, model_completion=True))

    print(f"\nCf coefficients:")
    for cf in coeffs_f:
        print(f"{cf} =", m.eval(cf, model_completion=True))

    print(f"\nV coefficients:")
    for v in coeffs_v:
        print(f"{v} =", m.eval(v, model_completion=True))
    
else:
    print("unsat: no solution exists for given degree ", d)
end_solve = time.time()
print("Solve time:", end_solve - start_solve)

Constraint construction time: 5.36606502532959
unsat: no solution exists for given degree  1
Solve time: 0.055844783782958984
